# WIT-metrics

Jupyter notebook

Calculate inundation summary metrics from Geoscience Australia Wetland Insights Tool (WIT) data (csv files)

[https://knowledge.dea.ga.gov.au/data/product/dea-wetlands-insight-tool-ramsar-wetlands/](https://knowledge.dea.ga.gov.au/data/product/dea-wetlands-insight-tool-ramsar-wetlands/)

[Dunn, B., Ai, E., Alger, M.J. et al. Wetlands Insight Tool: Characterising the Surface Water and Vegetation Cover Dynamics of Individual Wetlands Using Multidecadal Landsat Satellite Data. Wetlands 43, 37 (2023). https://doi.org/10.1007/s13157-023-01682-7](https://link.springer.com/article/10.1007/s13157-023-01682-7)

## Lineage

This code was derived from some initial code Geoscience Australia contributed to an MDBA project "BWS Vulnerabilities"
Changes since include:

* modernised code for Pandas 2.0+, Python 3.11 
* batch input of multiple WIT CVS in a folder (currently the ANAEv3 WIT output includes 270,653 polygons, each with its own csv file)
* multiple processor pool support to speed execution when running on a PC workstation
* linear interpolation of the observations dates to daily data to improve estimates of inundation duration, the interpolated data permits estimation of monthly stats
* some bug fixes in the inundation event metrics that were required when using the interpolated data
* output formatting

## Dependencies

     * A folder containing WIT csv (obtained for the BWS Priorities Project from Geoscience Australia for each ANAE polygon > 1Ha)

## Optional

     * A shapefile (or equivalent) that contains the area that the WIT result was run over.
  
     
## Background

The WIT data are generated by DEA with given wetland polygons and stored in a database on NCI. The data can be dumped into a csv when required. This notebook provides a way in computing temporal statistics (metrics) from the WIT csv.

## WIT Data definition

* WIT csv data files provide the following metrics for each polygon unit

      feature_id: The unique identifier for the polygon
      date: time of observation
      bs: percentage of bare soil
      npv: percentage of non photosynthetic vegetation
      pv: percentage of green/photosynthetic vegetation
      wet: percentage of wetness
      water: percentage of water
      pc_missing: the proportion of missing pixels in the polygon (cloud cover, satellite sensor issues)

## Description

This notebook uses existing WIT data to compute metrics.

* First we load the existing WIT csv data from a saved csv location
* Then we compute the metrics for all polygons and output the results to CSV files.  The input CSV files are processed in "batches" that are spread across multiple CPU cores.  When execution is complete the various batch outputs are merged together into single result files that contain metrics for every CSV feature ID (e.g. ANAE polygons)

The following files are created:

* **RESULT_WIT_yearly_metrics**: min, max, mean, median of each WIT metric per calendar year
* **RESULT_WIT_event_threshold**: for the BWS project we defined an "event" as exceeding the median [water+wet] - this file is read in to calculate the event metrics but could be replaced with user selected values if custom thresholds were wanted or the routine that generates it could be altered to change the threshold formulaically.
* **RESULT_WIT_time_since_last_inundation**: number of days since the inundation event threshold was exceeded
* **RESULT_WIT_inundation_metrics**: this is a join of the RESULT_WIT_ANAE_event_time and RESULT_WIT_ANAE_event_stats

If the optional debug_event_stats is set True then two intermediate files that are joined to make "RESULT_WIT_ANAE_inundation_metrics" are also saved:

* **RESULT_WIT_event_times**: start and end time, duration, duration of preceding gap (the dry period)
* **RESULT_WIT_event_stats**: for each event calculates the area of the polygon that was wet using the combination water+wet

### Processing Environment

Python 3.11.11

install requirements

```pip install -r requirements.txt```

### Execution

load this ```wit-metrics.ipynb``` into a Jupyter iPython environment and step through the notebook.  It will export the configuration entries to a python file ```config.py``` and will export and overwrite the code module ```wit_metrics_worker.py``` that runs outside of jupyter to speed up the code with parallel processing across available CPU cores

### Alternative method of execution in pure Python

the wit_metrics_worker.py module that is exported from the notebook (a copy is included in repository) can be run by itself.  First edit the config.py and then run wit_metrics_worker.py from the command line. 

```python wit_metrics_worker.py```
    
## Contact

Dr Shane Brooks
https://brooks.eco

![Brooks.eco logo](brooks-logo.png "Brooks Ecology & Technology")


# User Defined Parameters
are written to config.py to be read into python scripts 

In [28]:
%%writefile config.py

from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict
import os
import multiprocessing as mp




@dataclass(frozen=True)
class WITMetricsConfig():
    BASE_DIR: Path = Path(__file__).parent
    INPUT_DIR: Path = BASE_DIR / "input"
    OUTPUT_DIR: Path = BASE_DIR / "output"
    LOG_DIR: Path = BASE_DIR / "log"
    #-----------------------------------------------------------------
    # Config parameters are written to a python module config.py which is
    # imported by the main processing worker script wit_metrics_worker.py
    #
    # This file is generated by the wit-metrics.ipynb notebook and should not be modified directly.
    # Instead, modify the parameters in the notebook and then run the notebook to generate the config.py file.
    #-----------------------------------------------------------------

    # shapefile: the shape file mentioned above to find the  and get their area
    # set to '' to disable area lookup
    POLYGON_PATH = INPUT_DIR / "shp/ANAEv3_test.shp"

    # shapefile field name that identifies each polygon -  the ANAEv3 UID geohash was used here.
    # The ANAE UID is also used in the naming convention for the CSV files
    POLY_UNIQUE_ID = "UID"

    # Path to folder that contains the WIT csv files to process.  When debugging providing a single file might be prudent.
    WIT_CSV_PATH = INPUT_DIR / "csv"

    #Only use WIT data where the pc_missing is less than the threshold (default is 0.1) i.e. >90% of the polygon was visible to satellites
    PC_MISSING_THRESHOLD = 0.1

    # csv feature_id - the WIT csv output files include a column 'feature_id' that in this case is the ANAE UID
    PKEY = "feature_id"

    # whether to interpolate the WIT observation dates (typically 10-50+ per year) to daily data (365 per year)
    # This is computationally expensive but improves estimates of inundation duration and time since last inundation.
    # Monthly WIT stats require interpolated daily data to infill missing records and will not be generated if interpolate_to_daily = False
    INTERPOLATE_TO_DAILY = True

    # set to True to save the interpolated daily WIT csv in a subfolder under the csv_files (unnecessary and take up a lot of space but good for debugging)
    SAVE_INTERPOLATED_CSV = False

    #monthly metrics files are too big for most computers when all metrics are used (e.g. just 4 metrics x 270,000 polygons x 450 months is a 5GB csv file)
    # specify a subset that will be joined together into the monthly result - must include ["feature_id","date", at-least-one-metric]\
    MONTHLY_SUBSET = None #do not prune
    # monthly_subset=[
    #         "feature_id",
    #         "date",
    #         "water_median",
    #         "wet_median",
    #         "pv_median",
    #         "npv_median",
    #         "bs_median",
    #         "count",
    #     ]

    # set to true to save intermediate data frames containing the event times and stats
    # these are saved in the working directory
    DEBUG_EVENT_TIMES = False

    # batchsize is the number of WIT csv files to include in each 'batch' that is processed by each single CPU core.
    # The code was designed to process several 100,000 polygons in small batches that fit into the computer memory
    # then glue all the batch results together at the end.
    # On a workstation with 16 cpu cores and 64MB RAM a batchsize of 100-200 worked well. With smaller number of CSV a batch size < total number of CSV allows the
    # calculations to be spread across multiple processors.
    # during processing the code will generate outputs for each batch then glue them together at the end.

    BATCH_SIZE = 100
    
    #tag prepended to final result files (zipped csv)
    TAG="RESULT"
    
    #Whether to zip the final result csv to save space (python/pandas can read the csv from the zips)
    ZIP_RESULT = False

# ###########################################################

wit_metrics_cfg = WITMetricsConfig()


Overwriting config.py


# write wit_metrics_worker.py python module to the specified working directory
* To speed up the processing of many CSV files (we initially processed 270,653 ANAE csv) we divide the work across multiple processor cores.  To achieve this the notebook calls an external worker module contains the routines for summarising the WIT CSV data


In [29]:
%%writefile wit_metrics_worker.py

"""
wit_metrics_worker.py

This file is generated by the wit-metrics.ipynb notebook and should not be modified directly.
If changes are required modify the code in the notebook and then run the notebook to generate wit_metrics_worker.py
(or if you are comfortable in pure python you can ditch the notebook entirely (see #2 below)

USAGE:
(1) From the wit_metrics  Jupyter notebook.    This python module is exported from the Jupyter notebook to the current working directory then imported into
multiple threads that speed up the code by spreading the work over available CPU cores.
This conveniently allows the python code of this worker module to be maintained in the jupyter environment and stored with the multiprocessing code

(2)  wit_metrics_worker.py is self contained and can be run as a regular python script without the notebook. It will read the configuration parameters from config.py so make sure
you edit config.py before executing wit_metrics_worker.py

"""

import os
import sys

import numpy as np
import pandas as pd
import fiona
import pyarrow.parquet as pq
from shapely import geometry


# import os  # is loaded above to set the working directory
import glob
import logging
from pathlib import Path
from tqdm import tqdm
import time
import multiprocessing
from itertools import repeat
# ------------------------------------------------------------------------------
# Path & config setup
# ------------------------------------------------------------------------------

current_dir = Path(__file__).resolve().parent
if (current_dir / "config.py").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

#from tools.dask import start_dask
#from tools.logging_setup import setup_logging
from config import wit_metrics_cfg as config

logger = logging.getLogger(__name__)

# ------------------------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------------------------

def shape_list(key, values, shapefile):
    """
    Yields features from a shapefile that match the given keys.

    Args:
        key (str): The property name in the shapefile to filter by.
        values (iterable): A list or set of values to match against the key.
        shapefile (str): Path to the shapefile.

    Yields:
        tuple: A tuple containing (key_value, feature_object).
    """
    values = set(values)
    with fiona.open(shapefile) as src:
        for feat in src:
            k = feat["properties"].get(key)
            if k in values:
                yield k, feat


def get_areas(features, pkey):
    """
    Calculates the area of each feature in hectares.

    Args:
        features (iterable): An iterable of (id, feature) tuples.
        pkey (str): The column name to use for the ID in the output DataFrame.

    Returns:
        pd.DataFrame: A DataFrame indexed by pkey containing the 'area' column,
                      or None if no features are provided.
    """
    rows = []
    for gid, feat in features:
        # Calculate area and convert to hectares (1e4 m^2 = 1 ha)
        area = geometry.shape(feat["geometry"]).area / 1e4
        rows.append((gid, area))
    if not rows:
        return None
    return pd.DataFrame(rows, columns=[pkey, "area"]).set_index(pkey)


# ------------------------------------------------------------------------------
# Aggregation helpers
# ------------------------------------------------------------------------------

def _add_combined_members(df, members):
    """
    Adds combined metric columns to the DataFrame (e.g., 'water+wet').

    Args:
        df (pd.DataFrame): The input DataFrame.
        members (list): A list of members to process. Can be strings or lists of strings.
                        If a list is provided (e.g., ['water', 'wet']), a new column
                        'water+wet' is created by summing them.

    Returns:
        pd.DataFrame: The DataFrame with additional combined columns.
    """
    for m in members:
        if isinstance(m, list):
            name = "+".join(m)
            if name not in df:
                df[name] = df[m].sum(axis=1)
    return df


def _resample_metrics(df, freq, pkey):
    """
    Standard resampling logic. 
    Calculates each stat independently and merges them at the end.
    
    Args:
        df (pd.DataFrame): Input DataFrame with a DatetimeIndex.
        freq (str): The frequency string for resampling (e.g., 'YS' for year start).
        pkey (str): The column name to group by (e.g., 'feature_id').

    Returns:
        pd.DataFrame: A DataFrame with resampled statistics, reset index.
    """
    # 1. Group and Resample
    g = df.groupby(pkey).resample(freq, include_groups=False)

    # 2. Get the count (as a DataFrame so we can concat to it)
    count_df = g.size().astype("int32").to_frame(name="count")

    # 3. Identify numeric columns
    numeric_cols = df.select_dtypes(include="number").columns

    # 4. Explicitly calculate each block
    stats_dfs = [count_df]
    stats_to_run = ["max", "min", "mean", "median"]

    for stat in stats_to_run:
        # Get the stat (e.g., g.max(), g.mean())
        # getattr(g[numeric_cols], stat)() is equivalent to g[numeric_cols].max()
        res_stat = getattr(g[numeric_cols], stat)()

        # Rename: 'water' becomes 'water_max'
        res_stat.columns = [f"{c}_{stat}" for c in res_stat.columns]

        stats_dfs.append(res_stat)

    # 5. Concatenate all stats horizontally
    # Since they all share the same (pkey, date) index, they align perfectly
    res = pd.concat(stats_dfs, axis=1)

    return res.reset_index()



# ------------------------------------------------------------------------------
# Annual / Monthly metrics
# ------------------------------------------------------------------------------

def annual_metrics(wit_data, members=None, pkey="feature_id"):
    """
    Computes annual statistics for WIT members using optimized single-pass resampling.

    Args:
        wit_data (pd.DataFrame): The input WIT data.
        members (list, optional): List of metrics to compute. Defaults to standard set.
        pkey (str, optional): Primary key column. Defaults to "feature_id".

    Returns:
        pd.DataFrame: The computed annual metrics.
    """
    chunk = int(wit_data["chunk"].iat[0])
    members = members or [
        "pv", "wet", "water", "bs", "npv",
        ["npv", "pv", "wet"], ["pv", "wet"], ["water", "wet"]
    ]

    # 1. Create a local copy to avoid side-effects on the main wit_data
    # Use errors='ignore' in case columns were already dropped by monthly_metrics
    df = (
        wit_data.drop(columns=["chunk", "pc_missing"], errors='ignore')
        .set_index("date")
    )

    # 2. Add composite columns (e.g., water+wet)
    df = _add_combined_members(df, members)

    # 3. Resample using the optimized single-pass _resample_metrics
    # "YS" = Year Start (Jan 1st)
    res = _resample_metrics(df, "YS", pkey)

    # 4. Insert a clean 'year' column for easier end-user filtering
    res.insert(1, "year", res["date"].dt.year)

    # 5. Save output
    out = os.path.join(config.OUTPUT_DIR, f"WIT_yearly_metrics{chunk}.parquet")
    write_batch_parquet(res, out)

    return res


def monthly_metrics(wit_data, members=None, pkey="feature_id"):
    """
    Computes monthly statistics for WIT members.

    Args:
        wit_data (pd.DataFrame): The input WIT data.
        members (list, optional): List of metrics to compute.
        pkey (str, optional): Primary key column.

    Returns:
        pd.DataFrame: The computed monthly metrics.
    """
    chunk = int(wit_data["chunk"].iat[0])
    members = members or [
        "pv", "wet", "water", "bs", "npv",
        ["npv", "pv", "wet"], ["pv", "wet"], ["water", "wet"]
    ]

    # Drop non-essential columns and set date as index for resampling
    df = wit_data.drop(columns=["chunk", "pc_missing"], errors='ignore').set_index("date")
    df = _add_combined_members(df, members)

    # Use the optimized resampler
    res = _resample_metrics(df, "MS", pkey)

    # Add helper columns for easy filtering later
    res.insert(1, "month", res["date"].dt.month)
    res.insert(1, "year", res["date"].dt.year)

    out = os.path.join(config.OUTPUT_DIR, f"WIT_monthly_metrics{chunk}.parquet")
    write_batch_parquet(res, out)
    return res


# ------------------------------------------------------------------------------
# Inundation detection (vectorised, no apply)
# ------------------------------------------------------------------------------

def _event_table(df, threshold):
    """
    Identifies inundation events where 'water+wet' exceeds a threshold.

    Args:
        df (pd.DataFrame): Input DataFrame with 'water+wet' and 'date' columns.
        threshold (float): The threshold value for inundation.

    Returns:
        pd.DataFrame: A DataFrame of events with start_date, end_date, duration, and gap.
    """
    wet = df["water+wet"].to_numpy()
    dates = df["date"].to_numpy()

    inundated = wet > threshold
    if not inundated.any():
        return pd.DataFrame(
            columns=["start_date", "end_date", "duration", "gap"]
        )

    # Find indices where state changes (edges)
    edges = np.diff(inundated.astype(int))
    starts = np.where(edges == 1)[0] + 1
    ends = np.where(edges == -1)[0]

    # Handle boundary conditions (start or end of time series)
    if inundated[0]:
        starts = np.r_[0, starts]
    if inundated[-1]:
        ends = np.r_[ends, len(inundated) - 1]

    start_dates = dates[starts]
    end_dates = dates[ends]

    # Calculate duration in days (inclusive)
    duration = (end_dates - start_dates).astype("timedelta64[D]").astype(int) + 1
    
    # Calculate gap in days from previous event end to current event start
    gaps = np.r_[0, (start_dates[1:] - end_dates[:-1] - np.timedelta64(1, "D"))
                 .astype("timedelta64[D]").astype(int)]

    return pd.DataFrame({
        "start_date": start_dates,
        "end_date": end_dates,
        "duration": duration,
        "gap": gaps
    })


# ------------------------------------------------------------------------------
# Inundation detection with robust stats
# ------------------------------------------------------------------------------

def inundation_metrics(
    wit_data,
    threshold,
    shapefile,
    skey,
    debug_event_times=False,
    pkey="feature_id",
):
    """
    Calculates metrics for inundation events for each feature.

    Args:
        wit_data (pd.DataFrame): Input WIT data.
        threshold (pd.DataFrame or float): Threshold(s) for inundation.
        shapefile (str): Path to shapefile for area lookup.
        skey (str): Key in shapefile to match feature IDs.
        debug_event_times (bool, optional): Whether to output debug event times. Defaults to False.
        pkey (str, optional): Primary key column. Defaults to "feature_id".

    Returns:
        pd.DataFrame: DataFrame containing inundation metrics for each event.
    """
    chunk = int(wit_data["chunk"].iat[0])

    df = wit_data[[pkey, "date", "water", "wet"]].copy()
    df["water+wet"] = df["water"] + df["wet"]

    # Area lookup
    area = None
    if os.path.isfile(shapefile):
        feats = shape_list(skey, df[pkey].unique(), shapefile)
        area = get_areas(feats, pkey)

    events = []
    event_times = [] 
    stats = []

    for gid, g in df.groupby(pkey, group_keys=False, sort=False):
        th = threshold.loc[gid].iat[0] if isinstance(threshold, pd.DataFrame) else threshold
        ev = _event_table(g, th)
        if ev.empty:
            continue

        # Add feature_id and threshold
        ev.insert(0, pkey, gid)
        ev.insert(1, "threshold", th)
        events.append(ev)

        # Optional debug table
        if debug_event_times:
            event_times.append(
                ev[[pkey, "threshold", "start_date", "end_date", "duration", "gap"]]
            )

        # Compute water metrics for all events
        gvals = g.set_index("date")["water+wet"]
        for _, r in ev.iterrows():
            max_wet = gvals.loc[r.start_date:r.end_date].max()
            mean_wet = gvals.loc[r.start_date:r.end_date].mean()
            stats.append({
                pkey: gid,
                "start_date": r.start_date,
                "max_water+wet": max_wet,
                "mean_water+wet": mean_wet,
                "max_wet_area": max_wet * area.loc[gid, "area"] if area is not None and gid in area.index else np.nan,
                "mean_wet_area": mean_wet * area.loc[gid, "area"] if area is not None and gid in area.index else np.nan,
            })

    if not events:
        return pd.DataFrame()

    # Combine event tables
    event_df = pd.concat(events, ignore_index=True)
    if stats:
        st = pd.DataFrame(stats)
        event_df = event_df.merge(st, on=[pkey, "start_date"], how="left")

    # Convert dates to simple date format
    event_df["start_date"] = pd.to_datetime(event_df["start_date"]).dt.date
    event_df["end_date"] = pd.to_datetime(event_df["end_date"]).dt.date

    # Write output
    out = os.path.join(config.OUTPUT_DIR, f"WIT_inundation_metrics{chunk}.parquet")
    write_batch_parquet(event_df, out)

    # Optional debug output
    if debug_event_times and event_times:
        event_df = pd.concat(event_times, ignore_index=True)
        event_df["start_date"] = pd.to_datetime(event_df["start_date"]).dt.date
        event_df["end_date"] = pd.to_datetime(event_df["end_date"]).dt.date
        write_batch_parquet(
            event_df,
            os.path.join(config.OUTPUT_DIR, f"WIT_event_times{chunk}.parquet")
        )
        assert (
            pd.read_parquet(os.path.join(config.OUTPUT_DIR, f"WIT_event_times{chunk}.parquet"))
            .groupby([pkey, "start_date", "end_date"])
            .size()
            .max() == 1
        )

    return event_df



def interpolate_daily(wit_data, pkey="feature_id"):
    """
    Interpolates WIT data to a daily frequency using a linear interpolation
  
    Args:
        wit_data (pd.DataFrame): Input WIT data.
        pkey (str, optional): Primary key column. Defaults to "feature_id".

    Returns:
        pd.DataFrame: Daily interpolated DataFrame.
    """

    df = wit_data.copy()
    df["date"] = pd.to_datetime(df["date"])

    out = []

    for gid, g in df.groupby(pkey, sort=False):
        g = g.set_index("date")

        # numeric columns
        numeric = (
            g.select_dtypes("number")
             .resample("D")
             .mean()              # collapse same-day duplicates
             .interpolate()
        )

        # object columns
        objects = (
            g.select_dtypes("object")
             .resample("D")
             .first()
             .ffill()
        )

        daily = objects.join(numeric)
        daily[pkey] = gid
        out.append(daily.reset_index())

    return pd.concat(out, ignore_index=True)




# ------------------------------------------------------------------------------
# Time since last inundation (correct for never-inundated)
# ------------------------------------------------------------------------------

def time_since_last_inundation(wit_data, wit_im, pkey="feature_id"):
    """
    Calculates the time (in days) since the last inundation event for each feature.

    Args:
        wit_data (pd.DataFrame): Input WIT data (used for date range).
        wit_im (pd.DataFrame): Inundation metrics DataFrame (contains event info).
        pkey (str, optional): Primary key column. Defaults to "feature_id".

    Returns:
        pd.DataFrame: DataFrame with 'timesincelast' column.
    """
    chunk = int(wit_data["chunk"].iat[0])

    span = (
        wit_data.groupby(pkey, group_keys=False)["date"]
        .agg(first_date="min", final_date="max")
        .reset_index()
    )

    if wit_im is not None and not wit_im.empty:
        last = (
            wit_im.groupby(pkey,group_keys=False)["end_date"]
            .max()
            .reset_index()
            .rename(columns={"end_date": "last_event"})
        )
        span = span.merge(last, on=pkey, how="left")
        span["last_event"] = pd.to_datetime(span["last_event"])
        span["timesincelast"] = (
            span["final_date"] - span["last_event"]
        ).dt.days
    else:
        span["timesincelast"] = (
            span["final_date"] - span["first_date"]
        ).dt.days

    span["timesincelast"] = span["timesincelast"].fillna(
        (span["final_date"] - span["first_date"]).dt.days
    )

    out = os.path.join(
        config.OUTPUT_DIR,
        f"WIT_time_since_last_inundation{chunk}.parquet",
    )
    write_batch_parquet(span, out)
    return span


def all_time_median(wit_data, members=[["water", "wet"]], pkey="feature_id"):
    """
    Computes the all-time median for specified members.

    Args:
        wit_data (pd.DataFrame): Input WIT data.
        members (list, optional): List of members to compute median for. 
                                  Defaults to [['water', 'wet']].
        pkey (str, optional): Primary key column. Defaults to "feature_id".

    Returns:
        pd.DataFrame: DataFrame of medians indexed by pkey.
    """
    chunk = int(wit_data["chunk"].iat[0])
    wit_df = wit_data.copy(deep=True).drop(columns=["date", "chunk", "pc_missing"])
    wit_df = _add_combined_members(wit_df, members)
    wit_median = wit_df.set_index(pkey).groupby(pkey).median()
    out_file = os.path.join(
        config.OUTPUT_DIR,
        f"WIT_event_threshold{chunk}.parquet",
    )
    write_batch_parquet(wit_median.reset_index(), out_file)
    return wit_median


def merge_batches(
    path, output_filenames, tag="RESULT", monthly_subset=["feature_id", "date", "water+wet_median"], labels_df=None, Labels_join_field="feature_id"
):
    """
    Merges batch output files into single result files.

    Args:
        path (str): Directory containing batch files.
        output_filenames (list): List of base filenames to merge.
        tag (str, optional): Prefix for the output file. Defaults to "RESULT".
        monthly_subset (list, optional): Columns to keep for monthly metrics to save space.
        labels_df (pd.DataFrame, optional): DataFrame to join with results (e.g., for labels).
        Labels_join_field (str, optional): Field to join labels on. Defaults to "feature_id".
    """
    if tag:
        tag = tag + "_"
    for fname in output_filenames:
        out_files = glob.glob(os.path.join(path, fname + "*.parquet"))
        if out_files:
            result_fname = f"{tag}{fname}.csv"
            print(f"Merging {len(out_files)} batch outputs into {result_fname} ...")
            dfs = []
            for out_file in tqdm(out_files):
                try:
                    df = pd.read_parquet(out_file)
                    if monthly_subset and "monthly" in fname:
                        dfs.append(df[monthly_subset])
                    else:
                        dfs.append(df)
                except Exception as ex:
                    print(f"Error reading file: {out_file} - {ex}")
                    raise
            if dfs:
                out_data = pd.concat(dfs)
                out_data.reset_index(drop=True, inplace=True)
                if labels_df is not None:
                    if Labels_join_field not in out_data.columns:
                        raise
                    try:
                        out_data = out_data.merge(labels_df, on=Labels_join_field)
                    except Exception as ex:
                        print(f"Error merging labels: {ex}")
                        raise

                compression = None
                if config.ZIP_RESULT:
                    compression={"method": "zip", "archive_name": result_fname}
                    result_fname = f"{tag}{fname}.zip"
                try:
                    out_data.round(4).to_csv(os.path.join(path, result_fname), index=False, compression=compression)
                except Exception as ex:
                    print(f"Error writing merged file: {result_fname} - {ex}")
                    raise


def delete_old_batch_outputs(path, output_filenames):
    """
    Deletes the individual batch results which are no longer required after they have been merged.

    Args:
        path (str): Directory path.
        output_filenames (list): List of filename patterns to delete.
    """
    for fname in output_filenames:
        out_files = glob.glob(os.path.join(path, "./" + fname + "*.parquet"))
        for out_file in out_files:
            try:
                os.remove(out_file)
            except:
                print("Error while deleting file : ", out_file)


def nice_time(s):
    """Formats seconds into a human-readable string."""
    return time.strftime("%H hours %M minutes %S seconds", time.gmtime(s))

# ------------------------------------------------------------------------------
# Batch loader
# ------------------------------------------------------------------------------

def load_batch(csv_files, chunk_id):
    """
    Loads, cleans, and combines a batch of CSV files.

    Performs the following:
    1. Filters data based on missing data threshold.
    2. Normalizes dates (removes time).
    3. Averages duplicate entries for the same feature and date.
    4. Sorts by feature and date.

    Args:
        csv_files (list): List of CSV file paths.
        chunk_id (int): ID for the current chunk/batch.

    Returns:
        pd.DataFrame: Processed DataFrame for the batch.
    """
    dfs = []

    for f in csv_files:
        df = pd.read_csv(f)

        # 1. Filter missing data
        df = df[df["pc_missing"] < config.PC_MISSING_THRESHOLD]

        # 2. Normalize dates immediately (strip time/timezone)
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()

        dfs.append(df)

    if not dfs:
        return None

    # 3. Combine all files in the batch
    wit_data = pd.concat(dfs, ignore_index=True)

    # 4. Collapse duplicates: Average numeric data by Feature and Date
    # This handles overlapping data and ensures one row per day per feature
    wit_data = (
        wit_data.groupby([config.PKEY, "date"], as_index=False)
        .mean(numeric_only=True)
    )

    # 5. Critical: Sort chronologically for downstream np.diff logic
    wit_data = wit_data.sort_values([config.PKEY, "date"]).reset_index(drop=True)

    # Required by worker API
    wit_data["chunk"] = chunk_id

    return wit_data


def write_batch_parquet(batch_df, batch_fname):
    """
    Write a batch DataFrame to a temp Parquet file.

    Args:
        batch_df (pd.DataFrame): DataFrame to write.
        batch_fname (str): Target filename.

    Returns:
        str: The filename written to.
    """
    try:
        tmp_out = Path(batch_fname).with_suffix(".tmp.parquet")
        batch_df.to_parquet(tmp_out, index=False)
        os.replace(tmp_out, batch_fname)
        logger.info(f"  Written batch output:  {batch_fname}")
    except Exception as e:
        logger.error(f"  Failed to write {batch_fname}: {e}")
        tmp_out.unlink(missing_ok=True)
        raise

    return batch_fname


# ------------------------------------------------------------------------------
# Batch processor (single process)
# ------------------------------------------------------------------------------

def process_batch(csv_files, chunk_id):
    """
    Orchestrates the processing of a single batch of files.

    1. Loads data.
    2. Interpolates to daily (optional).
    3. Computes monthly and annual metrics.
    4. Computes inundation metrics.
    5. Computes time since last inundation.

    Args:
        csv_files (list): List of CSV files to process.
        chunk_id (int): ID for the batch.
    """
    # 1. Load, Normalize, and Sort
    # Handles daily averaging and strips time components immediately
    wit_data = load_batch(csv_files, chunk_id)

    if wit_data is None or wit_data.empty:
        return

    # 2. Optional daily interpolation
    # Fills gaps between observations using linear interpolation
    if config.INTERPOLATE_TO_DAILY:
        wit_data = interpolate_daily(wit_data)

        if config.SAVE_INTERPOLATED_CSV:
            # Note: This can be slow for very large batches
            out_dir = os.path.join(config.OUTPUT_DIR, "csv_daily_interpolated")
            os.makedirs(out_dir, exist_ok=True)
            # Efficiently save grouped CSVs
            for fid, group in wit_data.groupby(config.PKEY, sort=False):
                group.to_csv(os.path.join(out_dir, f"{fid}.csv"), index=False)

    # 3. Monthly & Annual Metrics
    # Uses the single-pass _resample_metrics for high speed
    monthly_metrics(wit_data)
    annual_metrics(wit_data)

    # 4. Inundation threshold (All-time median)
    # Calculates the threshold used for event detection
    median_df = all_time_median(wit_data)
    # Extract the specific column needed for _event_table
    threshold_df = pd.DataFrame(median_df["water+wet"])

    # 5. Inundation Metrics (The Core Analysis)
    # Vectorized event detection and area-weighted statistics
    wit_im = inundation_metrics(
        wit_data,
        threshold=threshold_df,
        shapefile=config.POLYGON_PATH,
        skey=config.POLY_UNIQUE_ID,
        debug_event_times=config.DEBUG_EVENT_TIMES,
    )

    # 6. Time Since Last Inundation
    # Uses wit_data and the newly created wit_im to find the final dry gap
    time_since_last_inundation(wit_data, wit_im)

if __name__ == "__main__":

    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    output_filenames = [
        "WIT_yearly_metrics",
        "WIT_event_threshold",
        "WIT_inundation_metrics",
        "WIT_time_since_last_inundation",
        "WIT_event_times",
        "WIT_event_stats",
    ]

    if config.INTERPOLATE_TO_DAILY:
        output_filenames.append("WIT_monthly_metrics")

    # Cleanup old batch outputs
    delete_old_batch_outputs(config.OUTPUT_DIR, output_filenames)

    csv_list = glob.glob(os.path.join(config.WIT_CSV_PATH, "*.csv"))
    total_files = len(csv_list)

    if total_files == 0:
        raise RuntimeError(f"No CSV files found in {config.WIT_CSV_PATH}")

    # Conservative core usage (important for IO-bound workloads)
    CPU = max(1, multiprocessing.cpu_count() // 2)

    batch_size = config.BATCH_SIZE
    chunk_size = batch_size * CPU

    print(f"Processing with {CPU} worker processes")
    print(f"Found {total_files} WIT CSV files")
    print(f"Batch size per process: {batch_size}")

    start = time.process_time()

    # ------------------------------------------------------------------
    # Batch-level multiprocessing only
    # ------------------------------------------------------------------
    for j in tqdm(range(0, total_files, chunk_size), desc="Processing chunks"):
        mpbatch = csv_list[j : j + chunk_size]

        work = []
        for i in range(0, len(mpbatch), batch_size):
            batch_files = mpbatch[i : i + batch_size]
            chunk_id = j + i
            work.append((batch_files, chunk_id))

        with multiprocessing.Pool(processes=min(len(work), CPU)) as pool:
            pool.starmap(process_batch, work)

    print(
        f"{total_files} CSVs processed in "
        f"{time.strftime('%H:%M:%S', time.gmtime(time.process_time() - start))}"
    )

    # ------------------------------------------------------------------
    # Merge batch outputs
    # ------------------------------------------------------------------
    merge_batches(
        config.OUTPUT_DIR,
        output_filenames,
        monthly_subset=config.MONTHLY_SUBSET,
        tag=config.TAG
    )

    print("All batches merged successfully.")

    # Optional cleanup
    delete_old_batch_outputs(config.OUTPUT_DIR, output_filenames)



Overwriting wit_metrics_worker.py


# Load packages
Import Python packages that are used for the analysis.

Use standard import commands; some are shown below. 


In [30]:
import os
import sys
import glob

from tqdm.notebook import tqdm
import time
import multiprocessing


#uncomment if running in ArcGIS Pro environment and GDAL errors are encountered
# os.environ["GDAL_DATA"] = "C:/Program Files/ArcGIS/Pro/Resources/pedata/gdaldata"

sys.path.append(
    os.getcwd()
)  # appends cwd to path allowing python to find config.py and wit_metrics_worker.py in the notebook directory
print (os.getcwd())




d:\wit-metrics


# multiprocessing code

this is the loop that executes to compute the metrics for all CSV in the specified path using multithreaded workers


In [31]:
#-----------------------------------------------------------------
# Reload the config and wit_metrics_worker in case they were edited
#-----------------------------------------------------------------
try:
    del sys.modules['config']
    del sys.modules['wit_metrics_worker']
except:
    pass

from config import wit_metrics_cfg as config
import wit_metrics_worker as worker


#-----------------------------------------------------------------
# code is under if __name__ == "__main__": to permit multiprocessing in jupyter notebooks
#-----------------------------------------------------------------
if __name__ == "__main__":

    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    output_filenames = [
        "WIT_yearly_metrics",
        "WIT_event_threshold",
        "WIT_inundation_metrics",
        "WIT_time_since_last_inundation",
        "WIT_event_times",
        "WIT_event_stats",
    ]

    if config.INTERPOLATE_TO_DAILY:
        output_filenames.append("WIT_monthly_metrics")

    # Cleanup old batch outputs
    worker.delete_old_batch_outputs(config.OUTPUT_DIR, output_filenames)

    csv_list = glob.glob(os.path.join(config.WIT_CSV_PATH, "*.csv"))
    total_files = len(csv_list)

    if total_files == 0:
        raise RuntimeError(f"No CSV files found in {config.WIT_CSV_PATH}")
    
    print(f"Found {total_files} WIT CSV files in folder {config.WIT_CSV_PATH}")

Found 6 WIT CSV files in folder D:\wit-metrics\input\csv


## Setup multiprocessing environment

In [32]:

# Conservative core usage (important for IO-bound workloads)
CPU = min(total_files, multiprocessing.cpu_count() // 2)

batch_size = config.BATCH_SIZE
chunk_size = batch_size * CPU

print(f"Processing {total_files} WIT CSV using {CPU} worker processes")
print(f"Maximum batch size per process: {batch_size}")

    

Processing 6 WIT CSV using 6 worker processes
Maximum batch size per process: 100


In [33]:
start = time.process_time()

# ------------------------------------------------------------------
# Batch-level multiprocessing only
# ------------------------------------------------------------------
for j in tqdm(range(0, total_files, chunk_size), desc="Processing chunks"):
    mpbatch = csv_list[j : j + chunk_size]

    work = []
    for i in range(0, len(mpbatch), batch_size):
        batch_files = mpbatch[i : i + batch_size]
        chunk_id = j + i
        work.append((batch_files, chunk_id))

    with multiprocessing.Pool(processes=min(len(work), CPU)) as pool:
        pool.starmap(worker.process_batch, work)

print(
    f"{total_files} CSVs processed in "
    f"{time.strftime('%H:%M:%S', time.gmtime(time.process_time() - start))}"
)


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

6 CSVs processed in 00:00:00


In [34]:

# ------------------------------------------------------------------
# Merge batch outputs
# ------------------------------------------------------------------
worker.merge_batches(
    config.OUTPUT_DIR,
    output_filenames,
    monthly_subset=config.MONTHLY_SUBSET,
    tag=config.TAG
)

print("All batches merged successfully.")

   

Merging 1 batch outputs into RESULT_WIT_yearly_metrics.csv ...


100%|██████████| 1/1 [00:00<00:00, 118.60it/s]


Merging 1 batch outputs into RESULT_WIT_event_threshold.csv ...


100%|██████████| 1/1 [00:00<00:00, 426.90it/s]


Merging 1 batch outputs into RESULT_WIT_inundation_metrics.csv ...


...

Merging 1 batch outputs into RESULT_WIT_time_since_last_inundation.csv ...


100%|██████████| 1/1 [00:00<00:00, 287.68it/s]


Merging 1 batch outputs into RESULT_WIT_monthly_metrics.csv ...


100%|██████████| 1/1 [00:00<00:00, 101.95it/s]

All batches merged successfully.


In [35]:
#check the outputs look ok and that batches merged properly into RESULT_WIT_* files

# Optional cleanup
worker.delete_old_batch_outputs(config.OUTPUT_DIR, output_filenames)

### END